In [ ]:
# Import FAISS vector store, Document class, and tools for LLM-based document compression and retrieval

from langchain_community.vectorstores import FAISS
from langchain_core.documents import Document
from langchain.retrievers.document_compressors import LLMChainExtractor
from langchain.retrievers.contextual_compression import ContextualCompressionRetriever

In [ ]:
# Add project root to sys.path and import custom functions to load Mistral LLM and Mini embeddings

import os
import sys

sys.path.append(os.path.abspath(os.path.join(os.getcwd(), '../../..')))
from utils.load_llms import load_mistral
from utils.load_embeddings import load_embedding_mini

In [5]:
# Recreate the document objects from the previous data
docs = [
    Document(page_content=(
        """The Grand Canyon is one of the most visited natural wonders in the world.
        Photosynthesis is the process by which green plants convert sunlight into energy.
        Millions of tourists travel to see it every year. The rocks date back millions of years."""
    ), metadata={"source": "Doc1"}),

    Document(page_content=(
        """In medieval Europe, castles were built primarily for defense.
        The chlorophyll in plant cells captures sunlight during photosynthesis.
        Knights wore armor made of metal. Siege weapons were often used to breach castle walls."""
    ), metadata={"source": "Doc2"}),

    Document(page_content=(
        """Basketball was invented by Dr. James Naismith in the late 19th century.
        It was originally played with a soccer ball and peach baskets. NBA is now a global league."""
    ), metadata={"source": "Doc3"}),

    Document(page_content=(
        """The history of cinema began in the late 1800s. Silent films were the earliest form.
        Thomas Edison was among the pioneers. Photosynthesis does not occur in animal cells.
        Modern filmmaking involves complex CGI and sound design."""
    ), metadata={"source": "Doc4"})
]

In [ ]:
# Generate embeddings using the Mini model and create a FAISS vector store from the documents
embedding_model = load_embedding_mini()               # Load custom Mini embedding model
vectorstore = FAISS.from_documents(docs, embedding_model)  # Build FAISS vector store

In [ ]:
# Convert FAISS vector store into a base retriever to fetch top 5 documents
base_retriever = vectorstore.as_retriever(search_kwargs={"k": 5})

In [ ]:
# Initialize an LLM-based compressor using the Mistral model

llm = load_mistral()                        # Load Mistral LLM
compressor = LLMChainExtractor.from_llm(llm)  # Create LLMChainExtractor for document compression

In [9]:
# Create the contextual compression retriever
compression_retriever = ContextualCompressionRetriever(
    base_retriever=base_retriever,
    base_compressor=compressor
)

In [10]:
# Query the retriever
query = "What is photosynthesis?"
compressed_results = compression_retriever.invoke(query)

In [ ]:
# Print the content of each document retrieved and compressed by the contextual compression retriever
for i, doc in enumerate(compressed_results):
    print(f"\n--- Result {i+1} ---")
    print(doc.page_content)


--- Result 1 ---
Photosynthesis is the process by which green plants convert sunlight into energy.

--- Result 2 ---
The chlorophyll in plant cells captures sunlight during photosynthesis.

--- Result 3 ---
Photosynthesis does not occur in animal cells.

--- Result 4 ---
Photosynthesis is a process that occurs in plants, algae, and some bacteria. In this process, carbon dioxide, water, and sunlight are used to produce glucose and oxygen.
